In [0]:
# Parameters (all optional). Leave public_dir blank to derive it from this
# notebook's location + relative_public_path, so the notebook works in any clone.
dbutils.widgets.text("public_dir", "", "Output public/ dir (blank = derive)")
dbutils.widgets.text("relative_public_path", "../../client/public", "public/ relative to this notebook")
dbutils.widgets.text("brand_subdir", "brand", "Subdir for symbol SVGs")
dbutils.widgets.text("assets_csv", "/Workspace/Shared/brandfolder/assets.csv", "Brandfolder assets catalog")

In [0]:
# Download Databricks brand assets (symbols and favicons) from the brandfolder
# CDN into an app's public/ dir (default: ../../client/public from here).
#
# Run once on compute that has outbound internet access.
# After running, commit the downloaded files.
#
# Output location and catalog come from the Parameters cell widgets.
# CDN URLs are looked up by filename in the assets catalog (assets_csv).
# Plan reference: docs/plans/ui_brand_plan.md §4 (Assets/identity)

import urllib.request, os, hashlib, csv
from pathlib import Path
from urllib.parse import urlsplit, urlunsplit
from PIL import Image  # pillow is pre-installed on DBR

# --------------------------------------------------------------------------- #
# Configuration (from widgets; nothing user- or repo-specific is hard-coded)
# --------------------------------------------------------------------------- #
def _notebook_dir() -> Path:
    ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
    nb_path = ctx.notebookPath().get()
    if not nb_path.startswith("/Workspace"):
        nb_path = "/Workspace" + nb_path
    return Path(nb_path).parent

public_dir = dbutils.widgets.get("public_dir").strip()
if public_dir:
    APP_PUBLIC = Path(public_dir)
else:
    rel = dbutils.widgets.get("relative_public_path").strip()
    APP_PUBLIC = Path(os.path.normpath(_notebook_dir() / rel))

if not APP_PUBLIC.is_dir():
    raise FileNotFoundError(
        f"Output dir {APP_PUBLIC} does not exist. Set the public_dir widget "
        "or fix relative_public_path (this notebook won't create it, to avoid "
        "writing assets to the wrong place)."
    )
BRAND_DIR = APP_PUBLIC / dbutils.widgets.get("brand_subdir").strip()
ASSETS_CSV = Path(dbutils.widgets.get("assets_csv").strip())
print(f"Output: {APP_PUBLIC}\nBrand subdir: {BRAND_DIR}\nCatalog: {ASSETS_CSV}\n")

# Look up CDN URLs by filename in the brandfolder catalog. The catalog URLs
# carry ?auto=webp&format=png, which converts SVGs to PNG; strip the query to
# get the original file.
with open(ASSETS_CSV, newline="") as f:
    _catalog = {r["Filename"]: (r.get("Attachment CDN URL") or r.get("File URL") or "")
                for r in csv.DictReader(f) if r.get("Filename")}

def cdn_url(filename: str) -> str:
    url = _catalog.get(filename)
    if not url:
        raise KeyError(f"{filename} not found in {ASSETS_CSV}")
    parts = urlsplit(url)
    return urlunsplit((parts.scheme, parts.netloc, parts.path, "", ""))

SVG_FILES = ["databricks-symbol-color.svg", "databricks-symbol-light.svg"]
FAVICON_FILES = [
    "databricks-favicon-16x16-orange.png",
    "databricks-favicon-32x32-orange.png",
    "databricks-favicon-48x48-orange.png",
    "databricks-favicon-512x512-orange.png",
]
SVG_ASSETS = {f: cdn_url(f) for f in SVG_FILES}
FAVICON_ASSETS = {f: cdn_url(f) for f in FAVICON_FILES}

# --------------------------------------------------------------------------- #
# Download helpers
# --------------------------------------------------------------------------- #
def download(url: str, dest: Path) -> None:
    """Download url to dest, creating parent dirs. Prints size + SHA-256."""
    dest.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(url, str(dest))
    size = dest.stat().st_size
    sha = hashlib.sha256(dest.read_bytes()).hexdigest()[:16]
    print(f"  {dest.name:.<50} {size:>8,} bytes  sha256:{sha}")

# --------------------------------------------------------------------------- #
# 1. Download SVGs into brand/
# --------------------------------------------------------------------------- #
print("=== SVG symbols (brand/) ===")
for name, url in SVG_ASSETS.items():
    download(url, BRAND_DIR / name)

# --------------------------------------------------------------------------- #
# 2. Download source favicon PNGs into a temp location, then copy to public/
#    with the names index.html expects
# --------------------------------------------------------------------------- #
print("\n=== Favicon PNGs ===")
for name, url in FAVICON_ASSETS.items():
    download(url, APP_PUBLIC / name)

# --------------------------------------------------------------------------- #
# 3. Generate derived sizes the app needs
# --------------------------------------------------------------------------- #
print("\n=== Derived favicons ===")
src_512 = APP_PUBLIC / "databricks-favicon-512x512-orange.png"
img = Image.open(src_512)

DERIVED = {
    "favicon-16x16.png":   16,
    "favicon-32x32.png":   32,
    "favicon-48x48.png":   48,
    "favicon-192x192.png": 192,
    "favicon-512x512.png": 512,  # copy at original size
    "apple-touch-icon.png": 180,
}
for name, size in DERIVED.items():
    dest = APP_PUBLIC / name
    resized = img.resize((size, size), Image.LANCZOS)
    resized.save(str(dest), "PNG")
    sha = hashlib.sha256(dest.read_bytes()).hexdigest()[:16]
    print(f"  {name:.<50} {size}x{size}  sha256:{sha}")

# --------------------------------------------------------------------------- #
# 4. Copy symbol-color as favicon.svg (replacing the old AppKit placeholder)
# --------------------------------------------------------------------------- #
print("\n=== favicon.svg ===")
import shutil
fav_svg = APP_PUBLIC / "favicon.svg"
shutil.copy2(BRAND_DIR / "databricks-symbol-color.svg", fav_svg)
print(f"  favicon.svg ← {(BRAND_DIR / 'databricks-symbol-color.svg').relative_to(APP_PUBLIC)} ({fav_svg.stat().st_size:,} bytes)")

# --------------------------------------------------------------------------- #
# 5. Summary
# --------------------------------------------------------------------------- #
print("\n=== Final public/ brand files ===")
for p in sorted(APP_PUBLIC.rglob("*")):
    if p.is_file() and any(k in p.name.lower() for k in ["favicon", "symbol", "apple-touch", "databricks"]):
        print(f"  {p.relative_to(APP_PUBLIC)}  ({p.stat().st_size:,} bytes)")

print("\n✓ Done. Commit these files, then delete the old placeholders if they differ.")
print("  Next: verify favicon.svg is the Databricks symbol (not Lava 300 shapes).")
print(f"  Next: git add -A {APP_PUBLIC} && git diff --cached --stat")